# 🧩 Advanced Custom Module Plugins, Surgery & Extensibility in NeuralOps

This flagship tutorial demonstrates how researchers and ML engineers can extend **NeuralOps** with bespoke attention mechanisms and compression engines.

### Key Capabilities Covered:
1. **Custom Attention Registration (`@register_custom_attention`)**: Register custom PyTorch attention layers into the global 131-module ecosystem with rich architectural metadata.
2. **6-Point Contract Diagnostic Validation (`validate_module_contract`)**: Automatically verify shapes, gradient flow, introspection metadata, micro-healing compatibility, and checkpoint serialization.
3. **Seamless Model Surgery Injection (`replace_attention`)**: Transplant custom attention directly into standard HuggingFace transformer architectures (GPT-2, LLaMA, etc.).
4. **Custom Compression Plugin Registration (`@register_custom_compressor`)**: Build and register custom model compression / low-rank factorization algorithms into the 72-engine catalog.
5. **Composable Kernel Map Pipeline (`compose_attention`)**: Programmatically compose projections, kernel maps, and attention engines without boilerplate.
6. **Graceful 3-Level Fallback & Unregistration**: Cleanly manage dynamic runtime lifecycles and fallback guarantees.

```
       ┌────────────────────────────────────────────────────────┐
       │             User Custom Attention / Compressor         │
       └───────────────────────────┬────────────────────────────┘
                                   │
              ┌────────────────────┴───────────────────┐
              ▼                                        ▼
    @register_custom_attention               @register_custom_compressor
              │                                        │
    ┌─────────▼──────────────┐               ┌─────────▼──────────────┐
    │ 131 Attention Catalog  │               │  72 Compressor Catalog │
    └─────────┬──────────────┘               └─────────┬──────────────┘
              │                                        │
    ┌─────────▼──────────────┐               ┌─────────▼──────────────┐
    │  replace_attention()   │               │   create_compressor()  │
    └────────────────────────┘               └────────────────────────┘
```


In [1]:
import os
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import GPT2Config, GPT2LMHeadModel

import neuralops
from neuralops import (
    register_custom_attention,
    register_custom_compressor,
    validate_module_contract,
    replace_attention,
    create_compressor,
    compose_attention,
    unregister_custom_module,
    ATTENTION_REGISTRY,
    COMPRESSOR_REGISTRY,
)

print(f"NeuralOps Version: {neuralops.__version__}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")
print(f"Active Canonical Attentions: {len(ATTENTION_REGISTRY)}")
print(f"Active Canonical Compressors: {len(COMPRESSOR_REGISTRY)}")


/mnt/d/venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NeuralOps Version: 1.4.0
Device: NVIDIA GeForce RTX 4070 Laptop GPU
Active Canonical Attentions: 131
Active Canonical Compressors: 72


## 1. Register a Bespoke Attention Mechanism

Researchers often design custom subquadratic recurrence formulations. Here, we define a feature-map recurrent linear attention mechanism:
$$\mathbf{S}_t = \mathbf{S}_{t-1} + \phi(\mathbf{k}_t)^T \mathbf{v}_t$$
$$\mathbf{o}_t = \phi(\mathbf{q}_t) \mathbf{S}_t$$

Decorating the class with `@register_custom_attention` instantly binds it into the NeuralOps registry, making it accessible across surgery, evaluation, and benchmarking tools.


In [2]:
@register_custom_attention(
    "my_research_attn",
    family="linear-scan",
    time_complexity="O(T·d²)",
    memory_complexity="O(d²) state",
    math_equation="S_t = S_{t-1} + phi(k_t)^T v_t | o_t = phi(q_t) S_t",
    supports_causal=True,
    supports_bidirectional=True,
)
class MyResearchAttention(nn.Module):
    """Feature-mapped linear attention mechanism with ELU feature map."""
    def __init__(self, embed_dim: int = 64, num_heads: int = 4, **kwargs):
        super().__init__()
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        
        self.q = nn.Linear(embed_dim, embed_dim, bias=False)
        self.k = nn.Linear(embed_dim, embed_dim, bias=False)
        self.v = nn.Linear(embed_dim, embed_dim, bias=False)
        self.out = nn.Linear(embed_dim, embed_dim, bias=False)

    def _feature_map(self, x: torch.Tensor) -> torch.Tensor:
        return F.elu(x) + 1.0

    def forward(self, x: torch.Tensor, *args, **kwargs):
        # x shape: [B, T, D]
        B, T, D = x.shape
        H, d = self.num_heads, self.head_dim

        q = self.q(x).view(B, T, H, d).transpose(1, 2)  # [B, H, T, d]
        k = self.k(x).view(B, T, H, d).transpose(1, 2)  # [B, H, T, d]
        v = self.v(x).view(B, T, H, d).transpose(1, 2)  # [B, H, T, d]

        q = self._feature_map(q)
        k = self._feature_map(k)

        # Numerically stable linear attention matrix product
        scores = torch.matmul(q, k.transpose(-1, -2))   # [B, H, T, T]
        ctx = torch.matmul(scores, v)                    # [B, H, T, d]

        out = self.out(ctx.transpose(1, 2).contiguous().view(B, T, D))
        if kwargs.get("return_tuple", False):
            return (out, None)
        return out

print(f"Module registered! Is 'my_research_attn' present? {'my_research_attn' in ATTENTION_REGISTRY}")


2026-10-18 17:08:06,302 [INFO] Registered custom attention module 'my_research_attn' (MyResearchAttention)


Module registered! Is 'my_research_attn' present? True


## 2. Automated 6-Point Contract Diagnostic Validation

NeuralOps provides an industrial-grade validation suite: `validate_module_contract()`.
It exercises 6 critical structural and mathematical invariants:
1. **Instantiation**: Constructor parameter handling (`embed_dim`, `num_heads`).
2. **Forward Shape**: Tensor dimensionality conservation `[B, T, D]`.
3. **Backward Gradients**: Clean gradient backpropagation through trainable parameters without vanishing/exploding.
4. **Introspection Metadata**: Valid registration of asymptotic complexities and equations.
5. **Surgical Micro-Healing**: Ability to perform closed-form subspace alignment against a teacher representation.
6. **Persistence Roundtrip**: Serialization and deserialization state fidelity across disk checkpoints.


In [3]:
val_results = validate_module_contract("my_research_attn", embed_dim=64, num_heads=4)
print(f"Contract Diagnostic Status: {'PASSED [OK]' if val_results['all_passed'] else 'FAILED'}")
print(f"Checks Passed: {val_results['passed_count']} / {val_results['total_checks']}\n")

for check_name, check_data in val_results["checks"].items():
    print(f"  [PASS] {check_name}: {check_data['status']}")


2026-10-18 17:08:06,694 [INFO] Saved surgical checkpoint with 1 modified submodules to /tmp/neuralops_contract_df8q773e


2026-10-18 17:08:06,695 [INFO] Successfully loaded surgical model checkpoint from /tmp/neuralops_contract_df8q773e


Contract Diagnostic Status: PASSED [OK]
Checks Passed: 6 / 6

  [PASS] 1_instantiation: PASSED
  [PASS] 2_forward_shape: PASSED
  [PASS] 3_backward_gradients: PASSED
  [PASS] 4_introspection_metadata: PASSED
  [PASS] 5_surgical_micro_healing: PASSED
  [PASS] 6_persistence_roundtrip: PASSED


## 3. Surgical Model Injection

Now we transplant our custom module directly into a HuggingFace Transformer model using `replace_attention()`.
NeuralOps inspects the host model's computational graph and safely swaps out the attention modules with zero manual code modifications.


In [4]:
# Instantiate a test Transformer model (e.g. GPT-2 architecture)
cfg = GPT2Config(n_embd=64, n_head=4, n_layer=2, vocab_size=500)
host_model = GPT2LMHeadModel(cfg)

print("Pre-surgery layer 0 attention:", type(host_model.transformer.h[0].attn))

# Execute surgical attention replacement
transplanted_model = replace_attention(host_model, attention="my_research_attn")

print("Post-surgery layer 0 attention:", type(transplanted_model.transformer.h[0].attn))

# Execute a forward pass through the surgically modified model
input_tokens = torch.randint(0, 500, (2, 32))
with torch.no_grad():
    outputs = transplanted_model(input_tokens)

print(f"Model forward pass successful! Output logits shape: {outputs.logits.shape}")


[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 499), got 50256. This may result in unexpected behavior.


[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 499), got 50256. This may result in unexpected behavior.


2026-10-18 17:08:06,710 [INFO] Successfully injected 'my_research_attn' into 2 transformer blocks.


Pre-surgery layer 0 attention: <class 'transformers.models.gpt2.modeling_gpt2.GPT2Attention'>
Post-surgery layer 0 attention: <class 'neuralops.utils.replace_attention.UniversalAttentionWrapper'>
Model forward pass successful! Output logits shape: torch.Size([2, 32, 500])


## 4. Register a Custom Compression Engine (`@register_custom_compressor`)

NeuralOps unifies model compression alongside attention mechanisms.
Using `@register_custom_compressor`, you can implement custom rank reduction, SVD projection, or structured pruning routines.


In [5]:
@register_custom_compressor(
    "custom_svd_truncator",
    family="low-rank-pruning",
    time_complexity="O(T·d·r)",
    memory_complexity="O(d·r) parameter footprint",
    math_equation="W' = U_k @ diag(S_k) @ V_k^T",
)
class CustomSVDTruncator:
    """Demonstration compressor that performs truncated SVD parameter reduction."""
    def __init__(self, model: nn.Module, rank: int = 16, **kwargs):
        self.model = model
        self.rank = rank

    def compress(self, **kwargs):
        orig_params = 0
        compressed_params = 0
        for name, module in self.model.named_modules():
            if isinstance(module, nn.Linear):
                W = module.weight.data
                orig_params += W.numel()
                k = min(self.rank, min(W.shape[0], W.shape[1]))
                U, S, V = torch.svd(W)
                # Low-rank reconstructed projection
                module.weight.data = (U[:, :k] @ torch.diag(S[:k]) @ V[:, :k].t()).contiguous()
                compressed_params += (U.shape[0] * k + k + k * V.shape[0])

        meta = {
            "compressor": "custom_svd_truncator",
            "rank": self.rank,
            "original_parameters": orig_params,
            "compressed_parameters": compressed_params,
            "compression_ratio": compressed_params / orig_params if orig_params > 0 else 1.0,
        }
        return self.model, meta

print(f"Compressor registered! Is 'custom_svd_truncator' in COMPRESSOR_REGISTRY? {'custom_svd_truncator' in COMPRESSOR_REGISTRY}")


2026-10-18 17:08:06,720 [INFO] Registered custom compressor module 'custom_svd_truncator' (CustomSVDTruncator)


Compressor registered! Is 'custom_svd_truncator' in COMPRESSOR_REGISTRY? True


In [6]:
# Instantiate test network and apply the custom compressor
test_linear_net = nn.Sequential(
    nn.Linear(64, 64),
    nn.ReLU(),
    nn.Linear(64, 32),
)

compressor = create_compressor("custom_svd_truncator", model=test_linear_net, rank=8)
compressed_net, comp_meta = compressor.compress()

print("Compression completed successfully!")
print(f"  Original parameter size:   {comp_meta['original_parameters']}")
print(f"  Compressed parameter size: {comp_meta['compressed_parameters']}")
print(f"  Parameter reduction:       {(1 - comp_meta['compression_ratio']) * 100:.2f}%")


Compression completed successfully!
  Original parameter size:   6144
  Compressed parameter size: 1808
  Parameter reduction:       70.57%


## 5. Composable Attention Kernel Pipeline

NeuralOps features an algebraic kernel composition engine (`compose_attention`).
Instead of writing an entire attention class, you can compose:
- **Projection**: `dense`, `gated`, `chunked`, `separable`
- **Kernel Map**: Custom elementwise activations (e.g., `custom_gelu_map`)
- **Mask**: `causal`, `sliding_window`, `local`, `none`
- **Engine**: `sdpa`, `triton`, `recurrent`, `chunkwise`


In [7]:
@neuralops.register_composable_kernel("custom_gelu_map")
def custom_gelu_map(x: torch.Tensor) -> torch.Tensor:
    """Custom shifted GELU feature map: GELU(x) + 1.0."""
    return F.gelu(x) + 1.0

# Compose an attention module algebraically
composed_module = compose_attention(
    projection="dense",
    kernel_map="custom_gelu_map",
    mask="causal",
    engine="sdpa",
    embed_dim=64,
    num_heads=4,
)

x = torch.randn(2, 16, 64)
out = composed_module(x)
print(f"Composed Attention Module: {type(composed_module)}")
print(f"Output shape: {out.shape}")


Composed Attention Module: <class 'neuralops.core.composable.ComposableAttention'>
Output shape: torch.Size([2, 16, 64])


## 6. Dynamic Lifecycle & Clean Unregistration

Custom modules can be cleanly deregistered from global registries to preserve runtime cleanliness between test runs and dynamic deployments.


In [8]:
print(f"Before unregister: 'my_research_attn' in ATTENTION_REGISTRY? {'my_research_attn' in ATTENTION_REGISTRY}")
unregister_custom_module("my_research_attn")
print(f"After unregister:  'my_research_attn' in ATTENTION_REGISTRY? {'my_research_attn' in ATTENTION_REGISTRY}")

unregister_custom_module("custom_svd_truncator")
print(f"After unregister:  'custom_svd_truncator' in COMPRESSOR_REGISTRY? {'custom_svd_truncator' in COMPRESSOR_REGISTRY}")
print("\nAll Extensibility & Plugin lifecycle tests completed successfully [100% OK]!")


Before unregister: 'my_research_attn' in ATTENTION_REGISTRY? True
After unregister:  'my_research_attn' in ATTENTION_REGISTRY? False
After unregister:  'custom_svd_truncator' in COMPRESSOR_REGISTRY? False

All Extensibility & Plugin lifecycle tests completed successfully [100% OK]!
